# 01 — Data preparation

Builds the analysis-ready cohort tables for all three data sources and the
interaction feature matrices that every downstream modelling stage consumes.

| Source | Access | Rows read | Rows kept |
|---|---|---|---|
| NHANES 1999–2012 | Public (CDC) | 71,916 | 11,660 |
| KNHANES 2019–2021 | Public on application (KDCA) | 22,559 | 15,138 |
| Taiwan Biobank | Restricted (Academia Sinica) | 196,963 | 92,734 |

Participants are kept when they are older than 18, report no diabetes diagnosis,
and have no missing value in any retained variable. Taiwan Biobank additionally
requires a fasting duration of at least eight hours.

The insulin-resistance label is `HOMA-IR > 2.5`, where
`HOMA-IR = fasting insulin × fasting glucose / 405`. Taiwan Biobank does not
measure fasting insulin and therefore carries no label; it is used as an
external validation cohort, scored by a model trained on the other two.

All logic lives in `src/`; this notebook only calls it and checks the result.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import polars as pl

from src.data.io import processed_path, write_parquet
from src.data.knhanes import process_knhanes
from src.data.nhanes import process_nhanes
from src.data.twb import process_twb
from src.features.interactions import generate_interactions
from src.logging_utils import configure_logging

configure_logging(ROOT / "logs")

## Cohort tables

Each reader returns the shared schema: an identifier, `SEX`, `AGE`, body
measurements, blood chemistry, mean arterial pressure, and — where fasting
insulin is available — `HOMA-IR` and the binary `IR` label.

In [2]:
nhanes = process_nhanes()
write_parquet(nhanes, processed_path("NHANES_data.parquet"))
print(f"NHANES {nhanes.shape}, IR+ {nhanes['IR'].mean():.4f}")
nhanes.head()

2026-09-22 19:45:39 [INFO] src.data.nhanes: NHANES rows read: 71916


2026-09-22 19:45:39 [INFO] src.data.nhanes: NHANES rows after filters and null removal: 11660


NHANES (11660, 21), IR+ 0.4445


Release_No,DIABETES,SEX,AGE,BMI,TG,LDL_C,T_CHO,FASTING_GLUCOSE,FASTING_INSULIN,HBA1C,HDL_C,BODY_WAISTLINE,SGPT,SGOT,BUN,CREATININE,URIC_ACID,MAP,HOMA-IR,IR
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool
"""000002""",2.0,1.0,77.0,24.9,128.0,136.0,215.0,83.7,4.55,4.7,54.0,98.0,16.0,19.0,19.0,0.7,6.1,70.0,0.940333,false
"""000005""",2.0,1.0,49.0,29.1,347.0,168.0,279.0,99.9,13.65,5.5,42.0,99.9,28.0,22.0,16.0,0.8,6.8,96.0,3.367,true
"""000007""",2.0,2.0,59.0,29.39,62.0,127.0,245.0,85.6,9.72,5.8,105.0,90.7,15.0,19.0,10.0,0.6,4.3,95.0,2.0544,false
"""000010""",2.0,1.0,43.0,30.94,45.0,80.0,140.0,89.8,6.43,5.5,51.0,108.0,20.0,26.0,13.0,0.9,6.0,110.666667,1.425714,false
"""000012""",2.0,1.0,37.0,30.62,146.0,89.0,156.0,82.9,26.52,5.2,38.0,112.8,35.0,17.0,20.0,1.0,5.7,124.0,5.428415,true


In [3]:
knhanes = process_knhanes()
write_parquet(knhanes, processed_path("KNHANES_data.parquet"))
print(f"KNHANES {knhanes.shape}, IR+ {knhanes['IR'].mean():.4f}")
knhanes.head()

2026-09-22 19:45:39 [INFO] src.data.knhanes: KNHANES hn20_all.sas7bdat rows read: 7359


2026-09-22 19:45:39 [INFO] src.data.knhanes: KNHANES hn21_all.sas7bdat rows read: 7090


2026-09-22 19:45:40 [INFO] src.data.knhanes: KNHANES hn19_all.sas7bdat rows read: 8110


2026-09-22 19:45:40 [INFO] src.data.knhanes: KNHANES rows after filters and null removal: 15138


KNHANES (15138, 21), IR+ 0.2792


Release_No,DIABETES,SEX,AGE,BMI,TG,LDL_C,T_CHO,FASTING_GLUCOSE,FASTING_INSULIN,HBA1C,HDL_C,BODY_WAISTLINE,SGPT,SGOT,BUN,CREATININE,URIC_ACID,MAP,HOMA-IR,IR
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool
"""A801169401""",0.0,1.0,39.0,24.185489,73.0,91.4,150.0,84.0,5.0,5.5,44.0,85.8,23.0,26.0,14.0,1.16,5.8,91.333333,1.037037,false
"""A801169402""",0.0,2.0,39.0,17.935939,65.0,109.0,187.0,89.0,3.6,5.3,65.0,68.0,20.0,22.0,13.0,0.72,4.6,74.666667,0.791111,false
"""A801172802""",0.0,2.0,58.0,26.58997,171.0,107.8,208.0,109.0,14.9,6.2,66.0,82.0,33.0,28.0,21.0,0.78,5.3,80.666667,4.010123,true
"""A801177901""",0.0,1.0,56.0,23.682126,49.0,174.2,241.0,107.0,4.1,5.6,57.0,86.0,25.0,28.0,15.0,0.76,5.2,84.333333,1.08321,false
"""A801177902""",0.0,2.0,53.0,19.669421,85.0,175.0,270.0,90.0,3.4,5.7,78.0,70.7,16.0,25.0,18.0,0.67,4.1,80.666667,0.755556,false


In [4]:
twb = process_twb()
write_parquet(twb, processed_path("TWB_clinical_data.parquet"))
print(f"Taiwan Biobank {twb.shape}")
twb.head()

2026-09-22 19:45:41 [INFO] src.data.twb: TWB rows read: lab_info=196961 survey=196963 measure=196963


2026-09-22 19:45:41 [INFO] src.data.twb: TWB rows after merge: 196963


2026-09-22 19:45:41 [INFO] src.data.twb: TWB rows after filters and null removal: 92734


Taiwan Biobank (92734, 18)


Release_No,AGE,SEX,BODY_WAISTLINE,BMI,FASTING_GLUCOSE,HBA1C,HDL_C,LDL_C,T_CHO,TG,SGOT,SGPT,BUN,CREATININE,URIC_ACID,MET_ID,MAP
str,i64,i64,f64,f64,i64,f64,i64,i64,i64,i64,i64,i64,f64,f64,f64,str,f64
"""ABBU000002""",62,1,83.7,22.471193,95,5.7,39,115,187,158,25,20,14.0,0.86,6.0,"""""",94.666667
"""ABBU000005""",60,1,93.0,31.045509,97,5.6,63,128,208,108,30,33,13.1,0.71,4.1,"""""",83.666667
"""ABBU000007""",63,1,94.0,26.850431,97,5.7,45,85,149,97,28,39,21.6,0.79,7.0,"""""",111.833333
"""ABBU000008""",69,1,98.5,26.519743,95,5.6,68,102,188,89,23,14,13.6,0.91,5.0,"""""",110.0
"""ABBU000012""",65,1,95.5,27.609452,93,5.7,49,127,191,108,26,18,17.8,1.1,5.7,"""""",102.166667


## Interaction features

Every eligible clinical variable gains a square, a square root and a base-10
logarithm, and every pair gains a product and a ratio. Rows containing a
non-finite generated value are dropped.

`RACE` is a coarse ancestry grouping of the cohort: 1 for the US cohort
(NHANES) and 2 for the East Asian cohorts (KNHANES and Taiwan Biobank). The
function default is 1.

Both KNHANES variants are written, each to its own file, because the two
downstream uses need different ones: the feature ablation reads the `RACE = 1`
table and the cross-ethnic and pooled models read `RACE = 2`. Writing them
separately keeps each file's contents independent of the order the notebooks
happen to run in.

The ablation's choice cannot affect its results either way: `RACE` is constant
within a single cohort, and those runs use default hyperparameters, so no tree
can split on it and no column sampling reaches it.

In [5]:
FEATURE_SPECS = [
    ("NHANES_data.parquet", 1, "NHANES_race1_features.parquet"),
    ("KNHANES_data.parquet", 1, "KNHANES_race1_features.parquet"),
    ("KNHANES_data.parquet", 2, "KNHANES_race2_features.parquet"),
    ("TWB_clinical_data.parquet", 2, "TWB_race2_features.parquet"),
]

for source, race, target in FEATURE_SPECS:
    features = generate_interactions(pl.read_parquet(processed_path(source)), race=race)
    write_parquet(features, processed_path(target))
    print(f"{target:34s} {features.shape}")

2026-09-22 19:45:41 [INFO] src.features.interactions: Generating interactions from 16 base features


2026-09-22 19:45:41 [INFO] src.features.interactions: Generated 288 features; dropping rows with non-finite values


2026-09-22 19:45:41 [INFO] src.features.interactions: Feature table: 11660 rows x 310 columns


2026-09-22 19:45:41 [INFO] src.features.interactions: Generating interactions from 16 base features


2026-09-22 19:45:41 [INFO] src.features.interactions: Generated 288 features; dropping rows with non-finite values


...

2026-09-22 19:45:42 [INFO] src.features.interactions: Generating interactions from 14 base features


2026-09-22 19:45:42 [INFO] src.features.interactions: Generated 224 features; dropping rows with non-finite values


KNHANES_race2_features.parquet     (15138, 310)


2026-09-22 19:45:42 [INFO] src.features.interactions: Feature table: 92734 rows x 243 columns


TWB_race2_features.parquet         (92734, 243)


### Feature-set sizes

The modelling notebooks drop the identifier and every column derived from
`HOMA-IR`, `FASTING_INSULIN` or `DIABETES` before training, which is what keeps
the target out of the feature matrix. The four feature sets used in the ablation
then have the following sizes.

The set built from the nine baseline variables plus their interactions contains
**57** features — the nine originals, their 27 single-variable transforms and
the 21 pairwise combinations that survive.

In [6]:
matrix = pl.read_parquet(processed_path("KNHANES_race1_features.parquet")).drop(
    ["Release_No", "^.*HOMA-IR.*$", "^.*FASTING_INSULIN.*$", "^.*DIABETES.*$"]
)

baseline = matrix.select(
    ["AGE", "BMI", "FASTING_GLUCOSE", "HBA1C", "HDL_C", "IR", "RACE", "SEX", "TG", "T_CHO"]
)
extended = matrix.select(pl.all().exclude("^.*[mul|log|div|sqrt].*$"))
baseline_with_interactions = matrix.select(
    pl.all().exclude("^.*(" + "|".join(set(extended.columns) - set(baseline.columns)) + ").*$")
)

for label, frame in [
    ("baseline", baseline),
    ("baseline + interactions", baseline_with_interactions),
    ("extended", extended),
    ("extended + interactions", matrix),
]:
    print(f"{label:26s} {frame.width - 1:3d} features")

baseline                     9 features
baseline + interactions     57 features
extended                    17 features
extended + interactions    241 features
